# 1. Setup and Imports

In [ ]:
# %pip install -q openai-whisper jiwer transformers sentencepiece sacrebleu TTS

import os
import sys
import json
import subprocess
import time
import glob
import random
import shutil
from pathlib import Path
from typing import Dict, List, Optional, Tuple

# --- Standard Libraries ---
import numpy as np
import cv2  # OpenCV for video processing
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from tqdm.notebook import tqdm  # Progress bars

# ============================================================================
# KAGGLE ENVIRONMENT DETECTION
# ============================================================================
IS_KAGGLE = os.path.exists('/kaggle/input')
print(f"Running on Kaggle: {IS_KAGGLE}")

if IS_KAGGLE:
    # Kaggle paths
    KAGGLE_COMPONENTS_PATH = '/kaggle/input/components-used'
    KAGGLE_DATA_PATH = '/kaggle/input/muavic'  # MuAViC dataset path
    
    # Add components to path
    if os.path.exists(KAGGLE_COMPONENTS_PATH):
        sys.path.insert(0, KAGGLE_COMPONENTS_PATH)
        print(f"✓ Added Kaggle components path: {KAGGLE_COMPONENTS_PATH}")
    else:
        print(f"⚠️  Warning: Kaggle components path not found: {KAGGLE_COMPONENTS_PATH}")
    
    # Verify MuAViC dataset exists (optional - will be downloaded via HuggingFace)
    if os.path.exists(KAGGLE_DATA_PATH):
        print(f"✓ MuAViC data found: {KAGGLE_DATA_PATH}")
        # List available files
        video_files = glob.glob(f"{KAGGLE_DATA_PATH}/**/*.mp4", recursive=True)
        print(f"  - Video files: {len(video_files)}")
    else:
        print(f"ℹ️  Note: MuAViC will be downloaded via Hugging Face datasets")

# ============================================================================
# IMPORT CUSTOM COMPONENTS
# ============================================================================
components_loaded = {
    'asr': False,
    'mt': False,
    'tts': False,
    'preprocessing': False,
    'lipsync': False
}

# Try importing each component separately for better error handling
try:
    from asr_component import ASRComponent, extract_audio_from_video, print_transcription, export_to_srt
    components_loaded['asr'] = True
    print("✓ ASR component imported")
except Exception as e:
    print(f"⚠️  ASR component import failed: {e}")

try:
    from mt_component import MTComponent
    components_loaded['mt'] = True
    print("✓ MT component imported")
except Exception as e:
    print(f"⚠️  MT component import failed: {e}")

try:
    from tts_component import TTSComponent, export_tts_manifest
    components_loaded['tts'] = True
    print("✓ TTS component imported")
except Exception as e:
    print(f"⚠️  TTS component import failed: {e}")

try:
    from data_preprocessing import DataPreprocessor, save_processed_data
    components_loaded['preprocessing'] = True
    print("✓ Data preprocessing component imported")
except (ImportError, SyntaxError) as e:
    print(f"⚠️  Data preprocessing import failed: {e}")

try:
    from lipsync_component import LipsyncGenerator, HighResSpatioTemporalDiscriminator, LowResAudioVisualDiscriminator
    components_loaded['lipsync'] = True
    print("✓ Lipsync component imported")
except Exception as e:
    print(f"⚠️  Lipsync component import failed: {e}")

# Summary
print(f"\n{'='*60}")
print("Component Import Summary:")
print(f"{'='*60}")
for component, loaded in components_loaded.items():
    status = "✅ Loaded" if loaded else "❌ Failed"
    print(f"  {component.upper():15s} : {status}")

if not all(components_loaded.values()):
    print(f"\n⚠️  Some components failed to load.")
    print("   The notebook will continue, but some features may be unavailable.")
else:
    print("\n✅ All custom components imported successfully!")
print(f"{'='*60}")

# ============================================================================
# IMPORT TTS LIBRARY (for Fine-tuning)
# ============================================================================
try:
    from TTS.config import BaseAudioConfig, BaseDatasetConfig
    from TTS.trainer import Trainer, TrainerArgs
    from TTS.tts.configs.shared_configs import BaseTTSConfig
    from TTS.tts.configs.tacotron2_config import Tacotron2Config
    from TTS.tts.datasets import BaseDatasetManager
    from TTS.tts.models.tacotron2 import Tacotron2
    from TTS.utils.audio import AudioProcessor
    print("✓ Coqui TTS library imported successfully")
except ImportError:
    print("⚠️  Warning: Coqui TTS library not fully found or installed.")
    print("   TTS Fine-tuning section might not work.")
    print("   Install with: pip install TTS")

# ============================================================================
# SET RANDOM SEEDS
# ============================================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("\n" + "="*60)
print("✓ Setup and Imports completed")
print("="*60)

# 2. Configuration

Configure paths and parameters for the video dubbing pipeline.

In [ ]:
# ============================================================================
# DIRECTORY CONFIGURATION
# ============================================================================
BASE_DIR = Path("/kaggle/working") if os.path.exists('/kaggle') else Path(".")
VIDEO_DATA_DIR = BASE_DIR / "muavic_videos"
PROCESSED_DATA_DIR = BASE_DIR / "processed_pretrain"
MODEL_CHECKPOINT_DIR = BASE_DIR / "models"
OUTPUT_DIR = BASE_DIR / "output"
TTS_FINETUNE_OUTPUT_DIR = MODEL_CHECKPOINT_DIR / "tts_finetuned"
LIPSYNC_MODEL_DIR = MODEL_CHECKPOINT_DIR / "lipsync"

# Create directories
for dir_path in [VIDEO_DATA_DIR, PROCESSED_DATA_DIR, MODEL_CHECKPOINT_DIR, 
                 OUTPUT_DIR, TTS_FINETUNE_OUTPUT_DIR, LIPSYNC_MODEL_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

print(f"📁 Directories configured:")
print(f"   Videos: {VIDEO_DATA_DIR}")
print(f"   Models: {MODEL_CHECKPOINT_DIR}")
print(f"   Output: {OUTPUT_DIR}")

# ============================================================================
# DATASET CONFIGURATION - MuAViC
# ============================================================================
# Pre-training: 5 languages, 2 hours each = 10 hours total
PRETRAIN_LANGUAGES = ['en', 'es', 'fr', 'de', 'ru']
HOURS_PER_LANGUAGE = 2.0

# Demo: Source → Target translation
SOURCE_LANG = 'en'
TARGET_LANG = 'es'

print(f"\n? Languages: {', '.join([l.upper() for l in PRETRAIN_LANGUAGES])}")
print(f"   Test set: {HOURS_PER_LANGUAGE}h × {len(PRETRAIN_LANGUAGES)} = {HOURS_PER_LANGUAGE * len(PRETRAIN_LANGUAGES)}h")
print(f"   Demo: {SOURCE_LANG.upper()} → {TARGET_LANG.upper()}")

# ============================================================================
# MODEL CONFIGURATION
# ============================================================================
ASR_MODEL_SIZE = "medium"  # Whisper
MT_MODEL_NAME = "facebook/nllb-200-distilled-600M"  # NLLB
BASE_TTS_MODEL_PATH = "tts_models/en/vctk/tacotron2-DDC"  # Tacotron 2

print(f"\n⚙️  Models:")
print(f"   ASR: Whisper-{ASR_MODEL_SIZE}")
print(f"   MT:  NLLB-200-600M")
print(f"   TTS: Tacotron 2 + WaveNet")

# ============================================================================
# TRAINING HYPERPARAMETERS
# ============================================================================
BATCH_SIZE = 4
LEARNING_RATE_GEN = 5e-4
LEARNING_RATE_DISC = 1e-4
TTS_FINETUNE_STEPS = 5000
LIPSYNC_PRETRAIN_STEPS = 10000
LIPSYNC_FINETUNE_STEPS = 1000

print(f"\n🎯 Training:")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Lipsync pretrain: {LIPSYNC_PRETRAIN_STEPS:,} steps")
print(f"   Lipsync finetune: {LIPSYNC_FINETUNE_STEPS:,} steps")

# ============================================================================
# DEVICE
# ============================================================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\n💻 Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

print(f"\n✅ Configuration complete")
print(f"   Next: Run MuAViC curation (Cell 2.2)")

# 2.1 About MuAViC Dataset

**MuAViC** = Multilingual Audio-Visual Corpus from Facebook Research

- **Access**: Hugging Face `facebook/muavic`
- **Size**: ~1900 hours across 9 languages
- **Content**: TED talk videos with transcriptions
- **Languages**: en, es, fr, de, ru, pt, it, el, ar

**Why MuAViC?** Easier than VoxCeleb2 - no extraction needed, pre-labeled languages, higher quality.

In [ ]:
# ============================================================================
# CHECK EXISTING DATA
# ============================================================================

print("Checking for existing data...\n")

# Check video directory
if VIDEO_DATA_DIR.exists():
    mp4_files = list(VIDEO_DATA_DIR.glob("*.mp4"))
    if mp4_files:
        print(f"✓ Found {len(mp4_files)} videos in {VIDEO_DATA_DIR}")
        # Language breakdown
        for lang in PRETRAIN_LANGUAGES:
            lang_videos = [v for v in mp4_files if v.name.startswith(f"{lang}_")]
            if lang_videos:
                print(f"  {lang.upper()}: {len(lang_videos)} videos")
    else:
        print(f"📭 No videos yet in {VIDEO_DATA_DIR}")
else:
    print(f"📭 Video directory not created yet")

# Check processed data
if PROCESSED_DATA_DIR.exists():
    processed_items = list(PROCESSED_DATA_DIR.iterdir())
    if processed_items:
        print(f"\n✓ Found {len(processed_items)} processed items in {PROCESSED_DATA_DIR}")
    else:
        print(f"\n📭 No processed data yet")
else:
    print(f"\n📭 Processed data directory not created yet")

print(f"\n{'='*60}")
print("Next: Run Cell 2.2 to download and curate MuAViC videos")
print("="*60)

# 2.2 MuAViC Data Curation (10-Hour Test Set)

Download 2 hours per language (en, es, fr, de, ru) = 10 hours total

**Note**: MuAViC requires GitHub repository method - not available directly on Hugging Face Hub. See https://github.com/facebookresearch/muavic

In [ ]:
# ============================================================================
# 2.2 MuAViC Dataset Download
# Uses GitHub repository: https://github.com/facebookresearch/muavic
# Target: 2h × 5 languages = 10 hours total
# ============================================================================

print("="*70)
print("MuAViC Dataset Download")
print("="*70)

# Configuration
TARGET_LANGUAGES = ['en', 'es', 'fr', 'de', 'ru']
HOURS_PER_LANGUAGE = 2.0
ROOT_PATH = "/kaggle/working" if os.path.exists('/kaggle') else "."
MUAVIC_REPO_PATH = Path(ROOT_PATH) / "muavic"
MUAVIC_DATA_PATH = Path(ROOT_PATH) / "muavic"

print(f"\n🎯 Target: {HOURS_PER_LANGUAGE}h × {len(TARGET_LANGUAGES)} languages")
print(f"📁 Root: {ROOT_PATH}\n")

# Step 1: Install dependencies
print("Step 1: Installing dependencies...")
!apt-get update -qq && apt-get install -y -qq ffmpeg sox
!pip install -q yt-dlp youtube-dl pydub tqdm
print("✓ Dependencies installed\n")

# Step 2: Clone MuAViC repository
print("Step 2: Cloning MuAViC repository...")
if not MUAVIC_REPO_PATH.exists():
    !git clone https://github.com/facebookresearch/muavic.git {ROOT_PATH}/muavic
    print("✓ Repository cloned")
else:
    print("✓ Repository exists")

# Install MuAViC requirements
requirements_file = MUAVIC_REPO_PATH / "requirements.txt"
if requirements_file.exists():
    !pip install -q -r {requirements_file}
    print("✓ Requirements installed\n")

# Step 3: Download data per language
print("Step 3: Downloading videos from TED/TEDx")
print("⏱️  Time: ~30-60 min per language (~3-5h total)\n")

download_status = {}
for i, lang in enumerate(TARGET_LANGUAGES, 1):
    print(f"[{i}/{len(TARGET_LANGUAGES)}] {lang.upper()}...")
    
    # Check if already downloaded
    lang_dir = MUAVIC_DATA_PATH / lang
    if lang_dir.exists():
        video_files = list((lang_dir / "video").glob("*.mp4")) if (lang_dir / "video").exists() else []
        if len(video_files) > 10:
            print(f"  ✓ Already downloaded: {len(video_files)} videos")
            download_status[lang] = "exists"
            continue
    
    # Download using get_data.py
    try:
        import subprocess
        result = subprocess.run(
            ["python", "get_data.py", "--root-path", str(ROOT_PATH), "--src-lang", lang],
            cwd=str(MUAVIC_REPO_PATH),
            capture_output=True,
            text=True,
            timeout=3600
        )
        
        if result.returncode == 0:
            video_files = list((lang_dir / "video").glob("*.mp4")) if (lang_dir / "video").exists() else []
            print(f"  ✓ Downloaded: {len(video_files)} videos")
            download_status[lang] = "success"
        else:
            print(f"  ❌ Failed: {result.stderr[:200]}")
            download_status[lang] = "failed"
    except subprocess.TimeoutExpired:
        print(f"  ❌ Timeout")
        download_status[lang] = "timeout"
    except Exception as e:
        print(f"  ❌ Error: {e}")
        download_status[lang] = "error"

# Step 4: Create manifest
print(f"\n{'='*70}")
print("Step 4: Creating manifest")
print(f"{'='*70}\n")

curated_data = []
for lang in TARGET_LANGUAGES:
    lang_dir = MUAVIC_DATA_PATH / lang
    if lang_dir.exists():
        video_dir = lang_dir / "video"
        audio_dir = lang_dir / "audio"
        
        video_files = list(video_dir.glob("*.mp4")) if video_dir.exists() else []
        
        print(f"{lang.upper()}: {len(video_files)} videos ({download_status.get(lang, 'unknown')})")
        
        for video_path in video_files:
            curated_data.append({
                'language': lang,
                'duration': 300.0,  # ~5 min estimate
                'video_path': str(video_path),
                'audio_path': str(audio_dir / f"{video_path.stem}.wav"),
                'sample_id': video_path.stem,
                'transcript': ''
            })

if curated_data:
    # Save manifest
    MUAVIC_MANIFEST_PATH = OUTPUT_DIR / "muavic_pretrain_manifest.json"
    manifest_data = {
        'total_clips': len(curated_data),
        'total_duration_hours': len(curated_data) * 5 / 60,
        'languages': TARGET_LANGUAGES,
        'download_status': download_status,
        'clips': curated_data
    }
    
    with open(MUAVIC_MANIFEST_PATH, 'w') as f:
        json.dump(manifest_data, f, indent=2)
    
    print(f"\n{'='*70}")
    print(f"✅ SUCCESS: {len(curated_data)} clips ready")
    print(f"{'='*70}")
    print(f"Manifest: {MUAVIC_MANIFEST_PATH}")
    print(f"Next: Run Cell 2.3 to preprocess videos")
    
    pretrain_curated_data = curated_data
else:
    print(f"\n{'='*70}")
    print("❌ NO DATA DOWNLOADED")
    print(f"{'='*70}")
    print("Check network connectivity and try again")
    pretrain_curated_data = []

# 2.3 Preprocess MuAViC Videos

Process curated videos: extract frames, detect faces, extract landmarks, create masks.

**Note**: Processing 500-1000 clips takes 2-4 hours. This cell is resumable.

In [ ]:
# ============================================================================
# 2.3 Preprocess MuAViC Videos
# ============================================================================

print("="*70)
print("MuAViC Video Preprocessing")
print("="*70)

# Load manifest
if 'pretrain_curated_data' not in locals() or not pretrain_curated_data:
    MUAVIC_MANIFEST_PATH = OUTPUT_DIR / "muavic_pretrain_manifest.json"
    if MUAVIC_MANIFEST_PATH.exists():
        with open(MUAVIC_MANIFEST_PATH, 'r') as f:
            pretrain_curated_data = json.load(f).get('clips', [])
        print(f"✓ Loaded {len(pretrain_curated_data)} clips from manifest")
    else:
        print(f"❌ Manifest not found. Run Cell 2.2 first.")
        pretrain_curated_data = []

if not pretrain_curated_data:
    print("No data to process.")
else:
    # Initialize preprocessor
    print("\nInitializing preprocessor...")
    preprocessor = DataPreprocessor(output_size=256)
    print("✓ Preprocessor ready")
    
    processed_dirs = []
    failed = []
    skipped = 0
    
    print(f"\nProcessing {len(pretrain_curated_data)} clips...")
    print("(This may take 2-4 hours)\n")
    
    for idx, clip in enumerate(tqdm(pretrain_curated_data, desc="Processing")):
        try:
            sample_id = clip.get('sample_id', f"clip_{idx}")
            language = clip.get('language', 'unknown')
            video_path = Path(clip.get('video_path', ''))
            
            # Check if video exists
            if not video_path.exists():
                failed.append(clip)
                continue
            
            # Output directory
            video_stem = video_path.stem
            output_dir = PROCESSED_DATA_DIR / language / video_stem
            meta_file = output_dir / f"{video_stem}_meta.json"
            
            # Skip if already processed
            if meta_file.exists():
                processed_dirs.append(str(output_dir))
                skipped += 1
                continue
            
            output_dir.mkdir(parents=True, exist_ok=True)
            
            # Process video
            result = preprocessor.process_video(str(video_path))
            
            if result:
                # Save processed data
                save_processed_data(result, str(output_dir), preprocessor)
                processed_dirs.append(str(output_dir))
            else:
                failed.append(clip)
        
        except Exception as e:
            print(f"\n❌ Error processing {sample_id}: {e}")
            failed.append(clip)
        
        # Progress update
        if (idx + 1) % 20 == 0:
            print(f"\n  Progress: {idx+1}/{len(pretrain_curated_data)}")
            print(f"  Processed: {len(processed_dirs)}, Skipped: {skipped}, Failed: {len(failed)}")
    
    # Summary
    print(f"\n{'='*70}")
    print(f"Preprocessing Complete")
    print(f"{'='*70}")
    print(f"Total clips:      {len(pretrain_curated_data)}")
    print(f"Processed:        {len(processed_dirs) - skipped}")
    print(f"Skipped:          {skipped}")
    print(f"Failed:           {len(failed)}")
    print(f"Output:           {PROCESSED_DATA_DIR}")
    
    # Save manifest
    manifest_path = PROCESSED_DATA_DIR / "processed_manifest.json"
    with open(manifest_path, 'w') as f:
        json.dump(processed_dirs, f, indent=2)
    print(f"\n✓ Manifest saved: {manifest_path}")
    
    if failed:
        failed_path = OUTPUT_DIR / "preprocessing_errors.json"
        with open(failed_path, 'w') as f:
            json.dump(failed, f, indent=2)
        print(f"⚠️  Failed clips saved: {failed_path}")
    
    print(f"\n✅ Ready for lip-sync training!")
    print(f"   Use manifest: {manifest_path}")